<a href="https://colab.research.google.com/github/fyremael/TROPICA/blob/main/notebooks/02_benchmark_suite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TROPICA Benchmark Suite

Goal: run the reproducible evidence suite, inspect gate status, show every SVG dashboard, and package the result artifacts for review.

In [ ]:
import pathlib
import subprocess
import sys

repo_root = pathlib.Path.cwd()
if (repo_root / "pyproject.toml").exists():
    install_cmd = [sys.executable, "-m", "pip", "install", "-q", "-e", ".[real-tokenizers,dev]"]
else:
    install_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "control-delta-support-decoding[real-tokenizers,dev] @ git+https://github.com/fyremael/TROPICA.git",
    ]
subprocess.check_call(install_cmd)


In [ ]:
import json
import pathlib
import subprocess

artifact_dir = pathlib.Path("colab_benchmark_artifacts")
cmd = ["cdsd-report", "--artifacts", str(artifact_dir), "--jobs", "4"]
if pathlib.Path("tests").exists():
    cmd.append("--with-pytest")
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)
manifest = json.loads((artifact_dir / "report_manifest.json").read_text())
assert manifest["passed"], [gate for gate in manifest["gates"] if not gate["passed"]]
manifest["passed"], len(manifest["commands"]), len(manifest["gates"])


In [ ]:
from IPython.display import Markdown, display

failed = [gate for gate in manifest["gates"] if not gate["passed"]]
summary = [
    "| Metric | Value |",
    "| --- | ---: |",
    f"| Overall pass | {manifest['passed']} |",
    f"| Commands | {len(manifest['commands'])} |",
    f"| Gates | {len(manifest['gates'])} |",
    f"| Failed gates | {len(failed)} |",
]
display(Markdown("\n".join(summary)))


In [ ]:
from IPython.display import Markdown, SVG, display

visuals = [
    ("Experiment", "experiment_visuals.svg"),
    ("Tokenizer correctness", "tokenizer_correctness_visuals.svg"),
    ("Structured output", "structured_output_visuals.svg"),
    ("Model integration", "model_integration_visuals.svg"),
    ("Stress", "stress_visuals.svg"),
    ("Scale", "scale_visuals.svg"),
]
for title, filename in visuals:
    display(Markdown(f"## {title}"))
    display(SVG(filename=str(artifact_dir / filename)))


In [ ]:
import shutil

zip_path = shutil.make_archive("colab_benchmark_artifacts", "zip", artifact_dir)
print("Packaged artifacts:", zip_path)


Interpretation: the benchmark notebook is the audit surface. The manifest is the machine-readable truth, the Markdown index is the reviewer surface, and the SVGs make regressions visible across correctness, scale, stress, tokenizer, structured-output, and model-integration tracks.